# 02. Pipeline RAG passo a passo

Este notebook abre a caixa: segue **uma pergunta real** por cada etapa do pipeline e mostra o
que entra e o que sai de cada uma, exatamente com o código de produção (`src/voc_rag`).

| Etapa do enunciado | Onde está no código |
|---|---|
| Transformação da pergunta | `retrieval/query.py` |
| Representação vetorial / embeddings | `indexing/embeddings.py` |
| Recuperação semântica (e lexical) | `retrieval/retriever.py`, `indexing/vector_store.py`, `indexing/bm25_index.py` |
| Seleção dos conteúdos mais relevantes | `retrieval/rerank.py`, `retrieval/retriever.py` (limiar, deduplicação, MMR) |
| Construção do contexto | `generation/answer.py` |
| Geração da resposta | `generation/llm.py`, `generation/answer.py`, `pipeline.py` |
| Abstenção | `pipeline.py` (três camadas) |

In [1]:
import warnings
warnings.filterwarnings("ignore")
import json
import numpy as np
import pandas as pd

from voc_rag.generation.llm import LLMClient
from voc_rag.retrieval.query import analisa_pergunta
from voc_rag.retrieval.retriever import (RetrievalConfig, consultas_de, filtros_efetivos,
                                         get_kb, recupera, rrf)
from voc_rag.generation.answer import (SYSTEM, RespostaGerada, monta_contexto, monta_prompt,
                                       rotula_evidencias, valida_citacoes)
from voc_rag.pipeline import VoCPipeline, render_markdown

pd.set_option("display.max_colwidth", 110)
kb = get_kb(); kb.aquece()
llm = LLMClient()
cfg = RetrievalConfig()
PERGUNTA = "Quais reclamações relacionadas à entrega aparecem com maior frequência nas avaliações com nota baixa?"
print(f"Base: {len(kb.docs):,} documentos | embedding: {kb.modelo} ({kb.emb.shape[1]} dimensões)")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Base: 42,138 documentos | embedding: BAAI/bge-m3 (1024 dimensões)


## Etapa 1. Transformação da pergunta

Uma chamada ao LLM (saída JSON validada por Pydantic) converte a pergunta do gestor em
instruções de busca. O ponto central: a pergunta está em linguagem **analítica**
("quais reclamações aparecem com maior frequência"), os documentos em linguagem de
**cliente** ("comprei dia 10 e até hoje nada"). A reescrita aproxima os dois registros.

In [2]:
analise = analisa_pergunta(PERGUNTA, kb.categorias, llm)
print(json.dumps(analise.model_dump(exclude={"origem"}), ensure_ascii=False, indent=2))

{
  "escopo": "dentro",
  "motivo_escopo": "A pergunta trata de relatos e reclamações de clientes sobre entrega em avaliações com nota baixa, o que está contido na base de dados de avaliações.",
  "tipo": "panorama",
  "polaridade": "negativa",
  "filtros": {
    "nota_min": 1,
    "nota_max": 2,
    "sentimentos": [],
    "categorias": [],
    "entrega_status": [],
    "ano_mes_inicio": null,
    "ano_mes_fim": null,
    "ufs": [],
    "regioes": []
  },
  "consulta_busca": "atraso na entrega produto não chegou prazo não cumprido entregador não apareceu",
  "consultas_alternativas": [
    "demora para entregar pacote extraviado",
    "problema com a transportadora entrega atrasada"
  ],
  "avaliacao_hipotetica": "Até agora meu pedido não chegou, passou do prazo de entrega e o atendimento não resolve nada. Muito insatisfeito com o serviço de transporte."
}


In [3]:
f = filtros_efetivos(analise, cfg)
print("Filtros extraídos da pergunta:", analise.filtros.descricao())
print("Filtros efetivos na busca    :", f.descricao())
print(f"Documentos no recorte        : {f.mask(kb.docs).sum():,} de {len(kb.docs):,}")

Filtros extraídos da pergunta: nota de 1 a 2
Filtros efetivos na busca    : nota de 1 a 2
Documentos no recorte        : 10,845 de 42,138


**Leitura.** "nota baixa" virou `nota_max = 2` (filtro de metadado aplicado dentro do banco
vetorial), o tipo `panorama` vai ampliar o contexto e ligar o quadro de frequências, e a
consulta reescrita usa o vocabulário dos clientes. O LLM também propõe variações (multi-query)
e uma avaliação hipotética (HyDE); as duas foram medidas no notebook 03, **pioraram** a
recuperação (puxam temas vizinhos) e ficam desligadas no sistema final. Elas aparecem abaixo
só para comparação.

## Etapa 2. Representação vetorial

Pergunta e documentos passam pelo mesmo modelo de embedding (`BAAI/bge-m3`, multilíngue,
1024 dimensões, vetores normalizados). A similaridade é o cosseno. O teste abaixo mede quanto
a reescrita aproxima a consulta de avaliações que de fato falam de problema de entrega.

In [4]:
consultas = consultas_de(analise, cfg, PERGUNTA)   # sistema final: só a consulta reescrita
exemplos = kb.docs[kb.docs["texto"].str.contains("não recebi|atras", case=False, regex=True)
                   & (kb.docs["nota"] <= 2)].sample(200, random_state=42)
vd = kb.emb[kb.docs_pos.loc[exemplos["review_id"]].to_numpy()]
textos = {"pergunta crua": PERGUNTA, "consulta reescrita (usada)": analise.consulta_busca,
          **{f"variação {i+1} (desligada)": q for i, q in enumerate(analise.consultas_alternativas)},
          "avaliação hipotética, HyDE (desligada)": analise.avaliacao_hipotetica}
vq = kb.embedder.encode_queries(list(textos.values()))
sims = pd.DataFrame({
    "texto da consulta": list(textos.values()),
    "cosseno médio com 200 reclamações de entrega": (vq @ vd.T).mean(axis=1).round(3),
}, index=list(textos))
sims

,texto da consulta,cosseno médio com 200 reclamações de entrega
pergunta crua,Quais reclamações relacionadas à entrega aparecem com maior frequência nas avaliações com nota baixa?,0.514
consulta reescrita (usada),atraso na entrega produto não chegou prazo não cumprido entregador não apareceu,0.700
variação 1 (desligada),demora para entregar pacote extraviado,0.623
variação 2 (desligada),problema com a transportadora entrega atrasada,0.612
"avaliação hipotética, HyDE (desligada)","Até agora meu pedido não chegou, passou do prazo de entrega e o atendimento não resolve nada. Muito insati...",0.684


## Etapa 3. Recuperação semântica e lexical

Cada consulta vai a dois índices:

* **denso** (ChromaDB, cosseno, filtro de metadado no `where`): captura sinônimos e paráfrases;
* **BM25** (stemming em português, negações preservadas): captura termos exatos
  ("correios", "nota fiscal") que o vetor às vezes dilui.

As listas são fundidas por **RRF** (Reciprocal Rank Fusion), que usa só a posição de cada
documento em cada lista: cosseno e BM25 não estão na mesma escala, a posição está. **A ordem
da fusão é a ordem final** (etapa 4 explica por quê).

In [5]:
where = f.to_chroma_where()
mask = f.mask(kb.docs)
qv = kb.embedder.encode_queries(consultas)
densas = kb.store.query(qv, k=5, where=where)
lex = kb.bm25.search(consultas[0], k=5, mask=mask)

def mostra(lista, rotulo):
    return pd.DataFrame([{"lista": rotulo, "pos": i + 1, "score": round(s, 3),
                          "texto": kb.docs.loc[d, "texto"]} for i, (d, s) in enumerate(lista)])

pd.concat([mostra(densas[0], "denso, consulta reescrita"), mostra(lex, "BM25, consulta reescrita")])

,lista,pos,score,texto
0,"denso, consulta reescrita",1,0.892,"atraso na entrega da mercadoria,não cumpre prazo determinado."
1,"denso, consulta reescrita",2,0.887,Atraso entrega. Não recebi o produto
2,"denso, consulta reescrita",3,0.884,entrega atrasada. o prazo terminou e nao recebi meu produto
3,"denso, consulta reescrita",4,0.881,"entrega atrasada,produto ainda não veio"
4,"denso, consulta reescrita",5,0.879,Não cumpriu com o prazo e o produto não chegou.
0,"BM25, consulta reescrita",1,8.056,não cumpre prazo. Não cumpriram prazo de entrega e nem avisaram o atraso.
1,"BM25, consulta reescrita",2,7.387,"atraso na entrega da mercadoria,não cumpre prazo determinado."
2,"BM25, consulta reescrita",3,6.827,Não cumpriu o prazo de entrega.
3,"BM25, consulta reescrita",4,6.827,NÃO CUMPRIU O PRAZO DE ENTREGA
4,"BM25, consulta reescrita",5,6.827,Não cumpriu o prazo de entrega!


In [6]:
res = recupera(PERGUNTA, analise, cfg, kb)
c = res.candidatos
print(f"Listas fundidas: {len(consultas)} densa(s) + {len(consultas)} lexical(is)")
print(f"Candidatas após RRF (top {cfg.n_rerank}): {len(c)}")
print(f"Recuperadas pelos dois índices: {(c['score_denso'].notna() & c['score_bm25'].notna()).sum()}"
      f" | só denso: {(c['score_denso'].notna() & c['score_bm25'].isna()).sum()}"
      f" | só BM25: {(c['score_denso'].isna() & c['score_bm25'].notna()).sum()}")
c.sort_values("pos_rrf")[["pos_rrf", "rrf", "score_denso", "score_bm25", "nota", "texto"]].head(8)

Listas fundidas: 1 densa(s) + 1 lexical(is)
Candidatas após RRF (top 80): 80
Recuperadas pelos dois índices: 17 | só denso: 32 | só BM25: 31


,pos_rrf,rrf,score_denso,score_bm25,nota,texto
0,1,0.032522,0.891783,7.387403,1,"atraso na entrega da mercadoria,não cumpre prazo determinado."
1,2,0.031099,0.868170,8.055669,1,não cumpre prazo. Não cumpriram prazo de entrega e nem avisaram o atraso.
2,3,0.030090,0.879161,6.740638,1,Não cumpriu com o prazo e o produto não chegou.
3,4,0.027894,0.887410,5.807443,1,Atraso entrega. Não recebi o produto
4,5,0.027778,0.864535,6.337264,1,"Não cumpre o prazo de entrega,até agora não recebi o meu produto."
5,6,0.027501,0.883696,5.769034,1,entrega atrasada. o prazo terminou e nao recebi meu produto
6,7,0.027480,0.865692,6.227702,2,Atraso. Produto não chegou no prazo
7,8,0.026830,0.852722,6.766582,2,Prazo de entrega não foi cumprido. Encomenda ainda não chegou.


## Etapa 4. Seleção dos conteúdos mais relevantes

1. **Reranker cross-encoder como porteiro** (`BAAI/bge-reranker-v2-m3`): lê o par (consulta,
   avaliação) junto e devolve relevância em [0, 1]. Pontuamos contra a consulta reescrita, não
   contra a pergunta analítica (a célula abaixo mostra por quê). Ele **não reordena**: no
   notebook 03, ordenar pelo reranker piorou a precisão, porque ele confunde temas vizinhos
   ("produto veio quebrado" no topo de uma pergunta sobre embalagem). Ele decide quem é
   relevante o bastante para entrar e quando não há evidência suficiente.
2. **Limiar 0,30** e **mínimo de 3 evidências** (escolhidos no notebook 03).
3. **Deduplicação**: "muito bom" de 50 clientes entra uma vez, com a contagem anotada.
4. **MMR**: entre as relevantes, prefere as que acrescentam algo diferente, mantendo a ordem da
   fusão.

In [7]:
textos_c = c["texto"].tolist()
contra_pergunta = kb.reranker.score(PERGUNTA, textos_c)
comparacao = pd.DataFrame({
    "reranker contra a pergunta analítica": np.quantile(contra_pergunta, [.25, .5, .75]).round(3),
    "reranker contra a consulta reescrita": np.quantile(c["relevancia"], [.25, .5, .75]).round(3),
}, index=["quartil 1", "mediana", "quartil 3"])
comparacao

,reranker contra a pergunta analítica,reranker contra a consulta reescrita
quartil 1,0.000,0.857
mediana,0.000,0.941
quartil 3,0.001,0.973


In [8]:
print(f"Candidatas: {len(c)} | acima do limiar ({cfg.limiar_relevancia}): {res.n_relevantes} | "
      f"no contexto após deduplicação e MMR: {len(res.selecionados)}")
res.selecionados[["pos_rrf", "relevancia", "repeticoes_no_topo", "nota", "entrega_status", "texto"]].head(12)

Candidatas: 80 | acima do limiar (0.3): 77 | no contexto após deduplicação e MMR: 30


,pos_rrf,relevancia,repeticoes_no_topo,nota,entrega_status,texto
0,1,0.994089,1,1,atrasado,"atraso na entrega da mercadoria,não cumpre prazo determinado."
1,2,0.947381,1,1,atrasado,não cumpre prazo. Não cumpriram prazo de entrega e nem avisaram o atraso.
2,3,0.996714,1,1,nao_entregue,Não cumpriu com o prazo e o produto não chegou.
3,4,0.957755,1,1,atrasado,Atraso entrega. Não recebi o produto
4,5,0.965641,1,1,cancelado,"Não cumpre o prazo de entrega,até agora não recebi o meu produto."
5,6,0.897156,1,1,atrasado,entrega atrasada. o prazo terminou e nao recebi meu produto
6,7,0.998622,1,2,atrasado,Atraso. Produto não chegou no prazo
7,8,0.995635,1,2,atrasado,Prazo de entrega não foi cumprido. Encomenda ainda não chegou.
8,9,0.963231,1,1,cancelado,"Não cumpriram o prazo de entrega,ainda não recebi o produto"
9,10,0.979076,1,2,atrasado,Entrega atrasada. Até o momento não foi entregue o produto


## Etapa 5. Construção do contexto

As evidências viram uma lista numerada `[R1]..[Rn]`. Cada uma leva os metadados que ajudam o
modelo a interpretar o texto (nota, situação e atraso da entrega, categoria, UF, mês). Em
perguntas de panorama entra também o **quadro de frequências**: contagem de aspectos na base
completa do recorte, para o modelo não confundir a amostra recuperada com a base inteira.

In [9]:
from voc_rag.themes.taxonomy import quadro_frequencias
evid = rotula_evidencias(res.selecionados)
quadro_df, quadro_txt = quadro_frequencias(kb.docs, res.filtros_aplicados, analise.polaridade) \
    if analise.tipo == "panorama" else (None, None)
prompt = monta_prompt(PERGUNTA, monta_contexto(evid), res.filtros_aplicados.descricao(),
                      res.n_universo, quadro_txt)
print(prompt[:3500], "\n[...]")

PERGUNTA DO GESTOR: Quais reclamações relacionadas à entrega aparecem com maior frequência nas avaliações com nota baixa?

RECORTE DA BASE: nota de 1 a 2 (10845 avaliações com comentário no recorte)

QUADRO DE FREQUÊNCIAS (base completa do recorte):
- Produto não recebido: ~4569 avaliações (42,1% das 10845 do recorte)
- Atendimento ruim: ~2800 avaliações (25,8% das 10845 do recorte)
- Atraso na entrega: ~2641 avaliações (24,4% das 10845 do recorte)
- Produto diferente do anunciado: ~2607 avaliações (24,0% das 10845 do recorte)
- Troca, devolução ou reembolso: ~2046 avaliações (18,9% das 10845 do recorte)
- Pedido incompleto: ~1906 avaliações (17,6% das 10845 do recorte)
- Defeito ou avaria: ~1540 avaliações (14,2% das 10845 do recorte)
- Qualidade abaixo do esperado: ~1438 avaliações (13,3% das 10845 do recorte)
- Embalagem ruim: ~192 avaliações (1,8% das 10845 do recorte)
(Estimativa de um classificador de aspectos com correção de erro; uma avaliação pode ter mais de um aspecto. Use c

## Etapa 6. Geração e validação

O LLM recebe as regras de fundamentação (prompt de sistema abaixo) e devolve JSON com resumo,
insights e os ids das evidências de cada insight. O código então **valida cada citação**
contra o contexto: id inexistente é removido e insight sem evidência é descartado.

In [10]:
print(SYSTEM)

Você é analista de Voice of Customer de um e-commerce. Responda à pergunta do gestor
usando EXCLUSIVAMENTE as avaliações de clientes fornecidas como evidências [R1]..[Rn] e,
quando houver, o QUADRO DE FREQUÊNCIAS.

Regras:
1. Toda afirmação sobre o que clientes relatam cita evidências no texto, uma marca por id:
   [R3][R7]. No texto, cite no máximo 4 evidências por frase (as mais representativas); em
   "evidencias", liste TODAS as que sustentam o insight (isso permite contar o apoio).
   Nunca cite um id que não exista.
2. Não use conhecimento externo. Não invente causas, números, percentuais, produtos,
   vendedores ou datas que não estejam nas evidências.
3. Números só podem vir (a) do QUADRO DE FREQUÊNCIAS, dizendo que se referem à base
   completa do recorte, ou (b) de contagens das próprias evidências ("4 das 15 avaliações
   recuperadas"). Nunca extrapole a amostra recuperada para a base inteira.
4. As evidências são uma AMOSTRA selecionada por relevância. Sem quadro de frequên

In [11]:
gerada = llm.generate(SYSTEM, prompt, RespostaGerada, etapa="geracao", pensamento="alto")
validada, relatorio = valida_citacoes(gerada, set(evid["rotulo"]))
print("Relatório de citações:", vars(relatorio))

Relatório de citações: {'citacoes_total': 18, 'citacoes_invalidas': 0, 'insights_total': 3, 'insights_descartados': 0, 'ids_invalidos': []}


In [12]:
from IPython.display import Markdown
Markdown(render_markdown(validada, len(evid)))

As reclamações relacionadas à entrega nas avaliações com nota baixa concentram-se no descumprimento de prazos e na ausência de recebimento dos produtos [R3][R7]. Conforme o quadro de frequências da base completa, o problema de produto não recebido atinge 42,1% dos casos, enquanto o atraso na entrega representa 24,4% [R1][R2]. Os clientes também apontam a falta de aviso prévio e a ausência de satisfação por parte dos fornecedores [R2][R15].

- **Produto não recebido**: O problema de produto não recebido é o mais frequente, representando 42,1% da base completa do recorte, com clientes relatando que a encomenda nunca chegou [R3][R8]. Vários compradores da amostra apontam que o item continua não entregue após o prazo determinado [R10][R14]. *(apoio: 20 de 30 evidências)*
- **Atraso na entrega e prazos descumpridos**: O atraso na entrega aparece em 24,4% das avaliações da base completa e é amplamente mencionado na amostra de avaliações [R1][R2]. Os consumidores criticam a falha da loja em cumprir as datas estabelecidas para a entrega [R20][R24]. *(apoio: 22 de 30 evidências)*
- **Ausência de comunicação sobre atrasos**: Os consumidores relatam insatisfação com a falta de aviso prévio por parte da loja quando ocorrem os problemas de entrega [R2][R15]. Há queixas específicas sobre a ausência de justificativas ou posições oficiais a respeito dos atrasos [R15][R21]. *(apoio: 3 de 30 evidências)*

*Limitações: As análises baseiam-se em uma amostra recuperada por relevância e no quadro de frequências da base completa do recorte de notas 1 e 2.*

In [13]:
citadas = sorted({e for i in validada.insights for e in i.evidencias}, key=lambda s: int(s[1:]))
evid[evid["rotulo"].isin(citadas)][["rotulo", "review_id", "nota", "texto"]].head(10)

,rotulo,review_id,nota,texto
0,R1,10467dc456e818c60bd750271b5d183e,1,"atraso na entrega da mercadoria,não cumpre prazo determinado."
1,R2,85a391b84d8e864de7190a2afb6f82e4,1,não cumpre prazo. Não cumpriram prazo de entrega e nem avisaram o atraso.
2,R3,a4f9f285b2ed89a616d93dea024eb31b,1,Não cumpriu com o prazo e o produto não chegou.
3,R4,9d6c1f490b2b4bfef33820b5679e2d82,1,Atraso entrega. Não recebi o produto
4,R5,0b89db97b78e8a32cf537f263f4aeb65,1,"Não cumpre o prazo de entrega,até agora não recebi o meu produto."
5,R6,1bfec3e913196c72c1ea06fc8816bd9c,1,entrega atrasada. o prazo terminou e nao recebi meu produto
6,R7,48b3ce236ea9d35a4850ea94ca9ca38f,2,Atraso. Produto não chegou no prazo
7,R8,6bb17113a061e2e8bb65b706278b4d75,2,Prazo de entrega não foi cumprido. Encomenda ainda não chegou.
8,R9,986820c39b1a0f1c99bf0592080078bd,1,"Não cumpriram o prazo de entrega,ainda não recebi o produto"
9,R10,66b7b7c200a3fa372cbd01606bba551f,2,Entrega atrasada. Até o momento não foi entregue o produto


**Rastreabilidade.** Cada `[R#]` da resposta leva a um `review_id` real da base da Olist; o
gestor pode auditar qualquer insight lendo as avaliações que o sustentam.

## Etapa 7. Quando não responder

Três camadas, da mais barata para a mais cara:

1. **Escopo** (análise da pergunta): pergunta fora da base é recusada antes de qualquer busca;
2. **Relevância** (reranker): nenhuma avaliação acima do limiar, ou menos que o mínimo de
   evidências, encerra sem chamar o gerador;
3. **Suficiência** (geração): o LLM declara `suficiente=false` quando as evidências tocam o
   tema mas não respondem (ex.: falam de entrega, mas nenhuma de entrega por drone).

In [14]:
pipe = VoCPipeline(llm=llm, kb=kb, aquecer=False)
for p in ["Qual foi o faturamento da Olist em 2018?",
          "Quais reclamações existem sobre pagamento com criptomoedas?",
          "O que os clientes acharam das entregas feitas por drone?"]:
    r = pipe.responder(p)
    print(f"\n>>> {p}\nstatus: {r.status}\n{r.markdown[:400]}")


>>> Qual foi o faturamento da Olist em 2018?
status: fora_escopo
**Fora do escopo da base de conhecimento.** A pergunta pede dados financeiros e de faturamento da empresa, que não constam nas avaliações de clientes.

A base contém apenas avaliações escritas por clientes da Olist entre 2016 e 2018 sobre suas compras (entrega, produto, atendimento, vendedor, embalagem, preço). Não respondo com conhecimento externo para não apresentar algo que os dados não sustent



>>> Quais reclamações existem sobre pagamento com criptomoedas?
status: sem_evidencia
**Não encontrei avaliações relevantes para responder com segurança.** Analisei 80 candidatas no recorte (nota de 1 a 3; 14.445 avaliações com comentário), e nenhuma atingiu a relevância mínima (0,30). Prefiro não responder a inventar uma conclusão sem suporte nos dados.

>>> O que os clientes acharam das entregas feitas por drone?
status: nao_respondida
**As evidências recuperadas não sustentam uma resposta.** As evidências recuperadas tratam apenas de entregas tradicionais de diversas categorias [R1][R2], não havendo nenhuma menção ou dado sobre entregas feitas por drone.
